In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import numpy as np
import pandas as pd

file_path = "/content/drive/MyDrive/eeg_seizure/EEG_Scaled_data.csv"

df = pd.read_csv(file_path, dtype=np.float32)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["target"].value_counts())
print("\nClass percentages:")
print((df["target"].value_counts(normalize=True) * 100).round(2))

Dataset shape: (11233, 36865)

Class distribution:
target
0.0    9799
1.0    1434
Name: count, dtype: int64

Class percentages:
target
0.0    87.23
1.0    12.77
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

df = df.dropna(subset=["target"]).copy()

X = df.drop(columns=["target"]).to_numpy(dtype=np.float32)
y = df["target"].to_numpy(dtype=np.int32)

# Replace non-finite input values with zero for this initial experiment.
# Check the dataset and document any such replacements.
X[~np.isfinite(X)] = 0.0

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape, np.bincount(y_train))
print("Validation:", X_val.shape, np.bincount(y_val))
print("Testing:", X_test.shape, np.bincount(y_test))

Training: (7863, 36864) [6859 1004]
Validation: (1685, 36864) [1470  215]
Testing: (1685, 36864) [1470  215]


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=16)

X_train_sel = selector.fit_transform(X_train, y_train)
X_val_sel = selector.transform(X_val)
X_test_sel = selector.transform(X_test)

selected_indices = selector.get_support(indices=True)

print("Selected feature indices:")
print(selected_indices)

print("Training shape:", X_train_sel.shape)
print("Validation shape:", X_val_sel.shape)
print("Testing shape:", X_test_sel.shape)

Selected feature indices:
[ 9622 11580 11581 11582 14878 14880 14881 15819 18697 19151 19152 20175
 20176 31058 32742 33657]
Training shape: (7863, 16)
Validation shape: (1685, 16)
Testing shape: (1685, 16)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
import time
import numpy as np

# Hardware-conscious Random Forest baseline
model = RandomForestClassifier(
    n_estimators= 25,
    max_depth=3,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=26,
    n_jobs=-1
)

# Train
start = time.time()
model.fit(X_train_sel, y_train)
training_time = time.time() - start

print(f"Training time: {training_time:.2f} seconds")
print("Number of trees:", len(model.estimators_))
print("Maximum tree depth:",
      max(tree.get_depth() for tree in model.estimators_))

Training time: 0.20 seconds
Number of trees: 25
Maximum tree depth: 3


In [ ]:
start = time.time()

val_prob = model.predict_proba(X_val_sel)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)

prediction_time = time.time() - start

print(f"Validation prediction time: {prediction_time:.4f} seconds")
print("Validation accuracy:",
      accuracy_score(y_val, val_pred))
print("Validation balanced accuracy:",
      balanced_accuracy_score(y_val, val_pred))
print("Validation ROC-AUC:",
      roc_auc_score(y_val, val_prob))
print("Validation PR-AUC:",
      average_precision_score(y_val, val_prob))

print("\nValidation classification report:")
print(classification_report(
    y_val, val_pred,
    target_names=["Non-seizure", "Seizure"],
    zero_division=0
))

print("Validation confusion matrix:")
print(confusion_matrix(y_val, val_pred))

Validation prediction time: 0.0137 seconds
Validation accuracy: 0.7875370919881306
Validation balanced accuracy: 0.769031798766018
Validation ROC-AUC: 0.8618082581869959
Validation PR-AUC: 0.5416096768654648

Validation classification report:
              precision    recall  f1-score   support

 Non-seizure       0.95      0.79      0.87      1470
     Seizure       0.35      0.74      0.47       215

    accuracy                           0.79      1685
   macro avg       0.65      0.77      0.67      1685
weighted avg       0.88      0.79      0.82      1685

Validation confusion matrix:
[[1167  303]
 [  55  160]]


In [ ]:
from sklearn.metrics import precision_score, recall_score

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.025):
    pred = (val_prob >= threshold).astype(int)

    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    balanced_acc = balanced_accuracy_score(y_val, pred)

    threshold_results.append(
        (threshold, precision, recall, balanced_acc)
    )

eligible = [r for r in threshold_results if r[2] >= 0.90]

if eligible:
    best = max(eligible, key=lambda r: (r[1], r[3]))
    print("Selected threshold with seizure recall >= 90%.")
else:
    best = max(threshold_results, key=lambda r: r[3])
    print("No threshold reached 90% recall.")
    print("Selected threshold with best validation balanced accuracy.")

best_threshold, precision, recall, balanced_acc = best

print(f"Threshold: {best_threshold:.3f}")
print(f"Validation precision: {precision:.3f}")
print(f"Validation recall: {recall:.3f}")
print(f"Validation balanced accuracy: {balanced_acc:.3f}")

Selected threshold with seizure recall >= 90%.
Threshold: 0.375
Validation precision: 0.253
Validation recall: 0.930
Validation balanced accuracy: 0.765


In [ ]:
import joblib

joblib.dump(
    {
        "model": model,
        "selector": selector,
        "threshold": float(best_threshold),
        "selected_indices": selected_indices
    },
    "/content/random_forest_eeg.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [ ]:
import json
import joblib
import numpy as np

saved = joblib.load("/content/random_forest_eeg.pkl")
rf_model = saved["model"]
threshold = saved["threshold"]
selected_indices = saved["selected_indices"]

forest_data = []

for tree_number, estimator in enumerate(model.estimators_):
    t = estimator.tree_

    forest_data.append({
        "tree_number": tree_number,
        "children_left": t.children_left.tolist(),
        "children_right": t.children_right.tolist(),
        "feature": t.feature.tolist(),
        "threshold": t.threshold.tolist(),
        "values": t.value.tolist(),
    })

export_data = {
    "threshold": float(threshold),
    "selected_indices": np.asarray(selected_indices).tolist(),
    "n_estimators": len(model.estimators_),
    "trees": forest_data,
}

with open("/content/random_forest_export.json", "w") as f:
    json.dump(export_data, f)

print("Trees exported:", len(forest_data))
print("Classification threshold:", threshold)
print("Selected input features:", len(selected_indices))
print("Export saved to /content/random_forest_export.json")

Trees exported: 25
Classification threshold: 0.3749999999999999
Selected input features: 16
Export saved to /content/random_forest_export.json


In [ ]:
import math
import joblib
import numpy as np

# Load the saved model
saved = joblib.load("/content/random_forest_eeg.pkl")
rf_model = saved["model"]
threshold = float(saved["threshold"])

N_FEATURES = 16
INPUT_SCALE = 256       # Q8.8 fixed-point
PROB_SCALE = 4095       # 12-bit probability scale

assert len(rf_model.estimators_) == 25
assert len(saved["selected_indices"]) == N_FEATURES

# Check that Q8.8 can represent all selected input values
if "X_train_sel" in globals():
    max_abs = float(np.max(np.abs(X_train_sel)))
    print("Maximum absolute selected feature:", max_abs)
    assert max_abs < 128, "Q8.8 input range exceeded; choose another format."

def signed_literal(value):
    value = int(value)
    if value < 0:
        return f"-16'sd{abs(value)}"
    return f"16'sd{value}"

def emit_tree_node(tree, node, indent, tree_id):
    pad = " " * indent

    left = tree.children_left[node]
    right = tree.children_right[node]

    # Leaf node: output quantized seizure probability
    if left == right:
        values = tree.value[node][0]
        p_seizure = float(values[1] / np.sum(values))
        p_quantized = int(np.rint(p_seizure * PROB_SCALE))
        p_quantized = max(0, min(PROB_SCALE, p_quantized))

        return f"{pad}p{tree_id} = 12'd{p_quantized};\n"

    # Tree feature index is relative to the 16 selected features
    feature = int(tree.feature[node])

    # Integer threshold for inputs rounded to Q8.8
    threshold_q = math.floor(
        float(tree.threshold[node]) * INPUT_SCALE
    )

    code = (
        f"{pad}if (x{feature} <= {signed_literal(threshold_q)}) begin\n"
    )
    code += emit_tree_node(tree, left, indent + 2, tree_id)
    code += f"{pad}end else begin\n"
    code += emit_tree_node(tree, right, indent + 2, tree_id)
    code += f"{pad}end\n"

    return code

# Module ports
ports = [
    f"input wire signed [15:0] x{i}" for i in range(N_FEATURES)
]
ports += ["output reg seizure_detected"]

lines = [
    "// Generated from the trained scikit-learn Random Forest",
    "// Inputs: 16 selected EEG values, signed Q8.8",
    "// Leaf probabilities: unsigned Q0.12-style integer scale 4095",
    "module rf_eeg(",
    "    " + ",\n    ".join(ports),
    ");",
    ""
]

# One probability register per tree
for i in range(len(rf_model.estimators_)):
    lines.append(f"reg [11:0] p{i};")

lines += [
    "reg [17:0] prob_sum;",
    ""
]

# Combinational decision logic for every tree
for i, estimator in enumerate(rf_model.estimators_):
    lines.append(f"always @* begin")
    lines.append(emit_tree_node(estimator.tree_, 0, 2, i).rstrip())
    lines.append("end")
    lines.append("")

# Sum all quantized tree probabilities and apply threshold
lines.append("always @* begin")
lines.append("    prob_sum = 18'd0;")

for i in range(len(rf_model.estimators_)):
    lines.append(f"    prob_sum = prob_sum + p{i};")

threshold_sum = math.ceil(
    threshold * PROB_SCALE * len(rf_model.estimators_)
)

lines.append(
    f"    seizure_detected = "
    f"(prob_sum >= 18'd{threshold_sum});"
)
lines.append("end")
lines.append("")
lines.append("endmodule")

verilog_code = "\n".join(lines)

with open("/content/rf_eeg.v", "w") as f:
    f.write(verilog_code)

print("Verilog generated successfully.")
print("Trees:", len(rf_model.estimators_))
print("Features:", N_FEATURES)
print("Threshold:", threshold)
print("RTL file: /content/rf_eeg.v")
print("Verilog lines:", len(lines))

Maximum absolute selected feature: 17.66200065612793
Verilog generated successfully.
Trees: 25
Features: 16
Threshold: 0.3749999999999999
RTL file: /content/rf_eeg.v
Verilog lines: 165


In [ ]:
from google.colab import files
files.download("/content/rf_eeg.v")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import numpy as np
import math
from google.colab import files

# Check that the trained model and test data are available
assert "rf_model" in globals(), "rf_model is missing. Load/train your model first."
assert "X_test_sel" in globals(), "X_test_sel is missing. Load your selected test features."
assert "threshold" in globals() or "best_threshold" in globals(), \
    "The decision threshold is missing."

# Use the threshold selected during validation
if "best_threshold" in globals():
    threshold = float(best_threshold)
else:
    threshold = float(threshold)

# Select 20 test samples, including both classes if possible
X = np.asarray(X_test_sel)
y = np.asarray(y_test) if "y_test" in globals() else None

if y is not None:
    chosen = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        chosen.extend(indices[:10].tolist())
    if len(chosen) < 20:
        chosen_set = set(chosen)
        chosen.extend(i for i in range(len(X)) if i not in chosen_set)
    chosen = chosen[:20]
else:
    chosen = list(range(min(20, len(X))))

assert X.shape[1] == 16, f"Expected 16 features, found {X.shape[1]}"
assert len(chosen) > 0, "No test samples available."

# Convert floating-point features to signed Q8.8 integers
def quantize_features(row):
    q = np.rint(np.asarray(row) * 256).astype(np.int64)
    if np.any(q < -32768) or np.any(q > 32767):
        raise ValueError("A feature exceeds the signed 16-bit range.")
    return q

# Emulate the tree traversal and probability accumulation in the RTL.
# This assumes the RTL uses floor(threshold*256), 0..4095
# leaf probabilities, and a summed threshold for all trees.
def rtl_reference(q):
    total = 0
    for estimator in rf_model.estimators_:
        tree = estimator.tree_
        node = 0

        while tree.children_left[node] != tree.children_right[node]:
            feature = tree.feature[node]
            split_q = math.floor(tree.threshold[node] * 256)
            if q[feature] <= split_q:
                node = tree.children_left[node]
            else:
                node = tree.children_right[node]

        values = tree.value[node][0]
        probability = values[1] / np.sum(values)
        probability_q = int(np.rint(probability * 4095))
        probability_q = max(0, min(4095, probability_q))
        total += probability_q

    required = math.ceil(
        threshold * 4095 * len(rf_model.estimators_)
    )
    return int(total >= required)

# Build the testbench
ports = [f"x{i}" for i in range(16)]
lines = [
    "`timescale 1ns/1ps",
    "module tb_rf_eeg_3;",
]

for port in ports:
    lines.append(f"reg signed [15:0] {port};")

lines.append("wire seizure_detected;")
lines.append("")
connections = ",\n    ".join(
    f".{port}({port})" for port in ports
)
lines += [
    "rf_eeg dut (\n    " + connections +
    ",\n    .seizure_detected(seizure_detected)\n);",
    "",
    "initial begin",
]

for test_no, idx in enumerate(chosen, start=1):
    q = quantize_features(X[idx])
    expected = rtl_reference(q)

    for i, value in enumerate(q):
        if value < 0:
            literal = f"-16'sd{abs(int(value))}"
        else:
            literal = f"16'sd{int(value)}"
        lines.append(f"    x{i} = {literal};")

    lines.append("    #10;")
    lines.append(
        f'    $display("Sample {test_no}: expected={expected}, '
        'actual=%0d", seizure_detected);'
    )
    lines.append(
        f"    if (seizure_detected !== 1'b{expected}) "
        f'$display("ERROR: Sample {test_no} mismatch");'
    )
    lines.append("")

lines += [
    '    $display("Testbench finished.");',
    "    $finish;",
    "end",
    "endmodule",
]

path = "/content/tb_rf_eeg_3.v"
with open(path, "w") as f:
    f.write("\n".join(lines) + "\n")

print("Created:", path)
print("Test vectors:", len(chosen))
print("Threshold:", threshold)
files.download(path)


Created: /content/tb_rf_eeg_3.v
Test vectors: 20
Threshold: 0.3749999999999999


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>